In [ ]:
! pip install -q openneuro-py mne mne-bids pyprep mne-icalabel autoreject

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mne, mne_bids, openneuro, os, glob, zipfile, re

from mne_bids import BIDSPath, read_raw_bids
from pyprep.find_noisy_channels import NoisyChannels
from mne_icalabel import label_components
from scipy.spatial.distance import cdist
from scipy.sparse import csr_matrix
from scipy import stats
from autoreject import get_rejection_threshold


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Main analyses

drive_preprocessed = '/content/drive/MyDrive/Meditation/preprocessed'
os.makedirs(drive_preprocessed, exist_ok = True)

drive_epochs = '/content/drive/MyDrive/Meditation/epochs'
os.makedirs(drive_epochs, exist_ok = True)

drive_tf = '/content/drive/MyDrive/Meditation/tf'
os.makedirs(drive_tf, exist_ok = True)

In [ ]:
# Files for main analysis

log_prep_file  = f"{drive_preprocessed}/preprocessing_log.csv"
answers_file   = f"{drive_preprocessed}/answers_from_text.csv"
rejection_file = f"{drive_epochs}/rejection_log.csv"
band_power_file = f"{drive_tf}/band_power_all.csv"


In [ ]:
# Robustness checks 2-4 (check 1 reuses the main band power)

robust_folders = {
  "ch2_iclabel05": '/content/drive/MyDrive/Meditation/robustness/ch_2',
  "ch3_paper_ica": '/content/drive/MyDrive/Meditation/robustness/ch_3',
  "ch4_no_ica":    '/content/drive/MyDrive/Meditation/robustness/ch_4',
}

for folder in robust_folders.values():
  os.makedirs(f"{folder}/epochs", exist_ok = True)
  os.makedirs(f"{folder}/tf", exist_ok = True)

1. Loading the data

In [ ]:
root = '/content/ds001787'

openneuro.download(dataset='ds001787', tag = '1.1.1', target_dir = root)

In [ ]:
def load_raws(root):

   channels = pd.read_csv(f'{root}/task-meditation_channels.tsv', sep = '\t')

   eeg_names = list(channels["name"][:64])

   ch_key = {'EEG': 'eeg', 'MISC': 'misc', 'GSR': 'gsr',
            'RESP': 'resp', 'PPG': 'bio', 'TEMP': 'temperature'}

   ch_types = dict(zip(channels['name'],channels['type'].map(ch_key)))

   paths = BIDSPath(root = root, task = 'meditation', datatype = 'eeg',
                 suffix = 'eeg', extension = '.bdf').match()

   raws = {(p.subject, p.session): read_raw_bids(p) for p in paths}

   for raw in raws.values():
     raw.rename_channels(dict(zip(raw.ch_names[:64], eeg_names)))
     raw.set_channel_types(ch_types)

   return raws



In [ ]:
participants = pd.read_csv(f"{root}/participants.tsv", sep='\t')
participants['subject'] = participants['participant_id'].str.replace('sub-', '')

In [ ]:
# sub-013/014: EEG matches subject 14's behavioral logs (identity unconfirmed)
excluded_subjects = ['013', '014']

# sub-007 ses-01: comb-like spectral artefact
excluded_sessions = [('007', '01')]

2.1. Preprocessing (only once, stored in Drive)

In [ ]:
def preprocessing(raw, key, drive_preprocessed):

  sub, ses = key
  raw.load_data()

  raw.set_montage(mne.channels.make_standard_montage('biosemi64'))

  raw.filter(l_freq = 1, h_freq = None)
  raw.notch_filter(freqs = 50 , picks = 'eeg', method = 'spectrum_fit')

  noisy = NoisyChannels(raw, random_state=123)
  noisy.find_all_bads()
  bads = noisy.get_bads()
  raw.info['bads'] = bads
  raw.interpolate_bads(reset_bads = True)

  raw.set_eeg_reference('average')

  fig = raw.compute_psd(picks = 'eeg', fmax = 128, tmax = 300).plot(show = False)
  fig.savefig(f"{drive_preprocessed}/sub-{sub}_ses-{ses}_psd_after.png")
  plt.close(fig)

  ica = mne.preprocessing.ICA(n_components = 0.99, method = 'infomax',
                              fit_params = dict(extended = True), random_state = 123)
  ica.fit(raw)
  labels = label_components(raw, ica, method = 'iclabel')
  exclude_labels = ['eye blink', 'muscle artifact', 'heart beat']
  ica.exclude = [l for l, (label, prob) in enumerate(zip(labels['labels'], labels['y_pred_proba']))
                 if label in exclude_labels and prob > 0.8]

  ica.save(f"{drive_preprocessed}/sub-{sub}_ses-{ses}_ica.fif", overwrite = True)

  prep_log_row = {'subject': sub, 'session': ses,
                  'n_bads': len(bads), 'bads': ','.join(bads),
                  'n_components_total' : ica.n_components_, 'n_excluded' : len(ica.exclude),
                  'excluded_labels' : [labels['labels'][l] for l in ica.exclude],}

  return raw, prep_log_row


In [ ]:
prep_log = (pd.read_csv(log_prep_file, dtype = {"subject": str, "session": str}).to_dict('records')
            if os.path.exists(log_prep_file) else [])

raws = load_raws(root)

for key in list(raws):

  sub, ses = key
  raw = raws.pop(key)

  ica_file = f"{drive_preprocessed}/sub-{sub}_ses-{ses}_ica.fif"
  if os.path.exists(ica_file):
    print(f'File {ica_file} already exists, skipping...')
    del raw
    continue

  print(f'Now processing subject {sub}, session {ses}')
  raw, prep_log_row = preprocessing(raw, key, drive_preprocessed)

  prep_log.append(prep_log_row)
  pd.DataFrame(prep_log).to_csv(log_prep_file, index = False)

  del raw

2.2. Epochs

2.2.1. Probe answers

In [ ]:
zip_path = f"{root}/code/MW_Current_TextFileBIDS.zip"

with zipfile.ZipFile(zip_path) as z:
    z.extractall(f"{root}/code/answers")

In [ ]:
text_dir = f"{root}/code/answers/MW_Current_TextFileBIDS"

def text_file(sub, ses):
    suffix = "" if int(ses) == 1 else f"_{int(ses)}"
    base = f"{text_dir}/sub{int(sub):02d}"
    for candidate in [f"{base}{suffix}_info.txt", f"{base}_{suffix}_info.txt"]:
        if os.path.exists(candidate):
            return candidate
    return candidate

def read_text_probes(path):
    probes = []
    for line in open(path):
        q = re.search(r"asked at time\s+([\d.]+)", line)
        if q:
            probes.append({"t_text": float(q.group(1))})
            continue
        k = re.search(r"Key (\d).*status (\d)", line)
        if k and probes:
            probes[-1][f"q{k.group(2)}"] = int(k.group(1))
    return pd.DataFrame(probes)

def align_probes(sub, ses, root, tol=0.5):
    path = text_file(sub, ses)
    if not os.path.exists(path):
        return None, {"subject": sub, "session": ses, "note": "no text file"}

    probes = read_text_probes(path)
    events = pd.read_csv(f"{root}/sub-{sub}/ses-{ses}/eeg/"
                         f"sub-{sub}_ses-{ses}_task-meditation_events.tsv", sep="\t")
    eeg_t = events.loc[events["value"] == 128, "onset"].values
    text_t = probes["t_text"].values

    shifts = (eeg_t[:, None] - text_t[None, :]).ravel()
    n_matches = [(np.abs(eeg_t[:, None] - (text_t + s)[None, :]) < tol).sum() for s in shifts]
    offset = shifts[np.argmax(n_matches)]

    gaps = np.abs(eeg_t[None, :] - (text_t + offset)[:, None])
    nearest = gaps.argmin(axis=1)
    error = gaps[np.arange(len(text_t)), nearest]
    matched = error < tol
    probes["onset"] = np.where(matched, eeg_t[nearest], np.nan)

    check = {"subject": sub, "session": ses, "offset": round(offset, 2),
             "n_text": len(text_t), "n_eeg": len(eeg_t), "n_matched": int(matched.sum()),
             "max_error": round(error[matched].max(), 3)}
    return probes, check

In [ ]:
all_probes, checks = [], []

for f in sorted(glob.glob(f"{root}/sub-*/ses-*/eeg/*_events.tsv")):
    parts = os.path.basename(f).split("_")
    sub = parts[0].replace("sub-", "")
    ses = parts[1].replace("ses-", "")
    if sub in excluded_subjects or (sub, ses) in excluded_sessions:
      continue

    probes, check = align_probes(sub, ses, root)
    checks.append(check)
    if probes is not None:
        probes["subject"], probes["session"] = sub, ses
        all_probes.append(probes)

checks = pd.DataFrame(checks)
print(checks.to_string())

answers = pd.concat(all_probes, ignore_index = True)
answers.to_csv(answers_file, index = False)

print(answers.groupby(["subject", "session"])[["q1", "q2"]]
      .apply(lambda d: d.isna().sum()).query("q1 > 0 or q2 > 0"))

In [ ]:
answers = pd.read_csv(answers_file, dtype = {"subject": str, "session": str})
answers = answers.merge(participants[["subject", "group"]], on="subject")

# Check 1: Table 1
per_subject = answers.groupby(["group", "subject"])[["q1", "q2", "q3"]].mean()
print(per_subject.groupby("group").mean().round(2))

# Check 2: Figure 3
answers["diff"] = answers["q1"] - answers["q2"]
pct = answers[answers["diff"] != 0].groupby(["group", "subject"])["diff"].apply(lambda d: (d > 0).mean() * 100)
print(pct.groupby("group").mean().round(1))

# Check 3: probes per subject
print(answers.groupby("subject").size().describe())

In [ ]:
def making_epochs(raw, key, answers):
    sub, ses = key
    probes = answers[(answers["subject"] == sub) & (answers["session"] == ses)].copy()
    probes = probes.dropna(subset=["onset", "q1", "q2"])
    probes = probes.rename(columns={"q1": "meditation", "q2": "mind_wandering", "q3": "drowsiness"})

    probes["condition"] = np.select(
        [probes["meditation"] > probes["mind_wandering"],
         probes["meditation"] < probes["mind_wandering"]],
        ["meditation", "mind_wandering"], default="tie")
    n_ties = int((probes["condition"] == "tie").sum())
    probes = probes[probes["condition"] != "tie"].reset_index(drop=True)

    if len(probes) == 0:
        return None, n_ties

    samples = (probes["onset"] * raw.info["sfreq"]).round().astype(int)
    codes = probes["condition"].map({"meditation": 1, "mind_wandering": 2})
    ev = np.column_stack([samples, np.zeros(len(probes), dtype=int), codes])
    epochs = mne.Epochs(raw, ev, event_id={"meditation": 1, "mind_wandering": 2},
                        tmin=-10.05, tmax=-0.05, baseline=None,
                        metadata=probes, preload=True, on_missing="warn")
    return epochs, n_ties


In [ ]:
raws = load_raws(root)

prep_log = (pd.read_csv(log_prep_file, dtype = {"subject": str, "session": str}).to_dict('records')
            if os.path.exists(log_prep_file) else [])
rejection_log = []

for key in list(raws):
  sub, ses = key
  if sub in excluded_subjects or (sub, ses) in excluded_sessions:
     continue

  print(f"Now epoching subject {sub}, session {ses}")
  raw = raws.pop(key)

  raw.load_data()
  raw.set_montage(mne.channels.make_standard_montage("biosemi64"))
  raw.filter(l_freq = 2, h_freq = None, method = 'iir',
             iir_params = dict(order = 6, ftype = 'butter'))
  raw.notch_filter(freqs = 50, picks = "eeg", method = "spectrum_fit")

  row = log[(log["subject"] == sub) & (log["session"] == ses)].iloc[0]
  raw.info["bads"] = row["bads"].split(",") if isinstance(row["bads"], str) else []
  raw.interpolate_bads(reset_bads = True)

  raw.set_eeg_reference("average")

  ica = mne.preprocessing.read_ica(f"{drive_preprocessed}/sub-{sub}_ses-{ses}_ica.fif")
  ica.apply(raw)

  epochs, n_ties = making_epochs(raw, key, answers)
  if epochs is not None:
    n_before = epochs.metadata["condition"].value_counts()
    reject = get_rejection_threshold(epochs, ch_types="eeg", random_state=123)
    epochs.drop_bad(reject=reject)
    n_after = epochs.metadata["condition"].value_counts()

    rejection_log.append({"subject": sub, "session": ses,
                    "threshold_uV": reject["eeg"] * 1e6,
                    **{f"{c}_before": n_before.get(c, 0) for c in ["meditation", "mind_wandering"]},
                    **{f"{c}_after": n_after.get(c, 0) for c in ["meditation", "mind_wandering"]}})

    if len(epochs) > 0:
        epochs.save(f"{drive_epochs}/sub-{sub}_ses-{ses}_epo.fif", overwrite=True)

  del raw, epochs

pd.DataFrame(rejection_log).to_csv(rejection_file, index = False)


In [ ]:
# Proportion of epochs kept per condition

rej = pd.read_csv(rejection_file, dtype = {"subject": str, "session": str})

for c in ["meditation", "mind_wandering"]:
    rej[f"{c}_kept"] = rej[f"{c}_after"] / rej[f"{c}_before"]
print(rej[["meditation_kept", "mind_wandering_kept"]].describe().round(2))

2. Replicate Brandemeyer and Delorme (2016)

In [ ]:
# Computing band power (once)

def compute_band_power(key, drive_epochs):
  sub, ses = key
  epochs = mne.read_epochs(f"{drive_epochs}/sub-{sub}_ses-{ses}_epo.fif")

  freqs = np.logspace(np.log10(1), np.log10(128), 100)
  n_cycles = 3 + (60 - 3) * (freqs - freqs[0]) / (freqs[-1] - freqs[0])
  tfr = epochs.compute_tfr('morlet', freqs = freqs, n_cycles = n_cycles,
                           picks = 'eeg', average = False, decim = 13)
  tfr.crop(tmin = -8.55, tmax = -1.55)
  spectra = tfr.get_data().mean(axis = -1)

  theta_idx = (freqs >= 4) & (freqs <= 7)
  alpha_idx = (freqs >= 9) & (freqs <= 11)
  theta = 10 * np.log10(spectra[:, :, theta_idx].mean(axis = -1)* 1e12)
  alpha = 10 * np.log10(spectra[:, :, alpha_idx].mean(axis = -1)* 1e12)

  n_epochs, n_channels = theta.shape
  band_power = pd.DataFrame({
      'epoch': np.repeat(np.arange(n_epochs), n_channels),
      'channel': np.tile(tfr.ch_names, n_epochs),
      'theta': theta.ravel(),
      'alpha': alpha.ravel(),
    })
  band_power['condition'] = epochs.metadata['condition'].values[band_power['epoch']]
  band_power['subject'], band_power['session'] = sub, ses

  return band_power

In [ ]:
files = sorted(glob.glob(f"{drive_epochs}/sub-*_ses-*_epo.fif"))
tf_log = []

for f in files:
  parts = os.path.basename(f).split('_')
  sub = parts[0].replace('sub-','')
  ses = parts[1].replace('ses-','')
  if sub in excluded_subjects or (sub, ses) in excluded_sessions:
    continue

  tf_log.append(compute_band_power((sub, ses), drive_epochs))

band_power_all = pd.concat(tf_log, ignore_index=True)
band_power_all.to_csv(band_power_file, index = False)


In [ ]:
# Epoch means per condition (meditation, mind-wandering)

band_power_all = pd.read_csv(band_power_file, dtype = {'subject': str, 'session': str})

n_trials = band_power_all.groupby(['subject', 'condition']).size().unstack(fill_value = 0) / 64
excluded = n_trials[(n_trials == 0).any(axis = 1)].index.tolist()
print('Excluded:', excluded)

subject_means = (band_power_all[~band_power_all['subject'].isin(excluded)]
                 .groupby(['subject', 'condition', 'channel'])[['theta', 'alpha']]
                 .mean().reset_index())

subject_means = subject_means.merge(participants[['subject', 'group']], on = 'subject')

In [ ]:
print(subject_means.groupby('group')['subject'].nunique())
print(n_trials.drop(index = excluded).sum(axis = 1).describe().round(1))

In [ ]:
# Meditation - mind-wandering per group x band

ch_names = mne.channels.make_standard_montage("biosemi64").ch_names

def difference_table(subject_means, group, band):
    data = subject_means[subject_means["group"] == group]
    wide = data.pivot_table(index="subject", columns=["condition", "channel"], values=band)
    return wide["meditation"][ch_names] - wide["mind_wandering"][ch_names]


In [ ]:
X_theta_exp    = difference_table(subject_means, "expert", "theta")
X_alpha_exp    = difference_table(subject_means, "expert", "alpha")
X_theta_novice = difference_table(subject_means, "novice", "theta")
X_alpha_novice = difference_table(subject_means, "novice", "alpha")

In [ ]:
pos = mne.channels.make_standard_montage("biosemi64").get_positions()["ch_pos"]
coords = np.array([pos[ch] for ch in ch_names])
dist = cdist(coords, coords)

for d in np.arange(0.020, 0.050, 0.001):
    print(round(d, 3), ((dist < d).sum(axis=1) - 1).mean())

adjacency = csr_matrix(dist < 0.047)

In [ ]:
def compare_conditions (x, adjacency, ch_names):

  threshold = stats.t.ppf(0.975, df=len(x) - 1)

  T_obs, clusters, cluster_pv, H0 = mne.stats.permutation_cluster_1samp_test(
    x.values, adjacency = adjacency, threshold = threshold,
    n_permutations = 2048, tail = 0, rng = 123)

  cluster_channels = [[ch_names[i] for i in c[0]]
                      for c, p in zip(clusters, cluster_pv) if p < 0.05]

  t_vals, p_vals = stats.ttest_1samp(x.values, 0)
  reject, p_fdr = mne.stats.fdr_correction(p_vals, alpha=0.05, method="negcorr")
  fdr_channels = [ch for ch, r in zip(ch_names, reject) if r]

  return {"T_obs": T_obs, "cluster_pv": cluster_pv,
          "cluster_channels": cluster_channels, "fdr_channels": fdr_channels}

In [ ]:
res_theta_exp = compare_conditions(X_theta_exp, adjacency, ch_names)
res_alpha_exp = compare_conditions(X_alpha_exp, adjacency, ch_names)
res_theta_novice = compare_conditions(X_theta_novice, adjacency, ch_names)
res_alpha_novice = compare_conditions(X_alpha_novice, adjacency, ch_names)

print(res_theta_exp['cluster_pv'], res_theta_exp['cluster_channels'], res_theta_exp['fdr_channels'])
print(res_alpha_exp['cluster_pv'], res_alpha_exp['cluster_channels'], res_alpha_exp['fdr_channels'])
print(res_theta_novice['cluster_pv'], res_theta_novice['cluster_channels'], res_theta_novice['fdr_channels'])
print(res_alpha_novice['cluster_pv'], res_alpha_novice['cluster_channels'], res_alpha_novice['fdr_channels'])


In [ ]:
results = {"main": {("expert", "theta"): res_theta_exp,
                    ("expert", "alpha"): res_alpha_exp,
                    ("novice", "theta"): res_theta_novice,
                    ("novice", "alpha"): res_alpha_novice}}

In [ ]:
# Figure: Expert topographies

info = mne.create_info(ch_names, sfreq=256, ch_types="eeg")
info.set_montage("biosemi64")

experts = subject_means[subject_means["group"] == "expert"]
avg = experts.groupby(["condition", "channel"])[["theta", "alpha"]].mean()

fig, axes = plt.subplots(2, 3, figsize=(10, 6))

for row, band in enumerate(["theta", "alpha"]):
    med = avg.loc["meditation", band].reindex(ch_names).values
    mw = avg.loc["mind_wandering", band].reindex(ch_names).values
    vlim = (min(med.min(), mw.min()), max(med.max(), mw.max()))

    mne.viz.plot_topomap(med, info, axes=axes[row, 0], show=False, vlim=vlim, cmap="jet")
    mne.viz.plot_topomap(mw, info, axes=axes[row, 1], show=False, vlim=vlim, cmap="jet")
    mne.viz.plot_topomap(med - mw, info, axes=axes[row, 2], show=False,
                         vlim=(-0.6, 0.6), cmap="jet")

    axes[row, 0].set_title(f"{band}: meditation")
    axes[row, 1].set_title(f"{band}: mind wandering")
    axes[row, 2].set_title(f"{band}: difference")
    print(band, "difference range (dB):", round((med - mw).min(), 2), "to", round((med - mw).max(), 2))

plt.tight_layout()
plt.show()

Robustness checks

In [ ]:
def run_tests(band_power_all, adjacency):

  n_trials = band_power_all.groupby(['subject', 'condition']).size().unstack(fill_value = 0) / 64
  excluded = n_trials[(n_trials == 0).any(axis = 1)].index.tolist()

  sm = (band_power_all[~band_power_all['subject'].isin(excluded)]
        .groupby(['subject', 'condition', 'channel'])[['theta', 'alpha']]
        .mean().reset_index()
        .merge(participants[['subject', 'group']], on = 'subject'))

  tests = {}
  for group in ["expert", "novice"]:
    for band in ["theta", "alpha"]:
      X = difference_table(sm, group, band)
      tests[(group, band)] = compare_conditions(X, adjacency, ch_names)
  print("Excluded:", excluded)
  return tests


def ica_on_epochs(epochs, key, out_dir):
  sub, ses = key
  ica = mne.preprocessing.ICA(n_components = 0.99, method = 'infomax',
                              fit_params = dict(extended = True), random_state = 123)
  ica.fit(epochs)
  labels = label_components(epochs, ica, method = "iclabel")
  exclude_labels = ['eye blink', 'muscle artifact', 'heart beat']
  ica.exclude = [i for i, (lab, p) in enumerate(zip(labels['labels'], labels['y_pred_proba']))
                 if lab in exclude_labels and p > 0.8]
  ica.save(f"{out_dir}/sub-{sub}_ses-{ses}_ica.fif", overwrite = True)
  ica.apply(epochs)
  return epochs, {'n_excluded': len(ica.exclude),
                  'n_samples_fit': len(epochs) * len(epochs.times)}


def robustness_epochs(check, epochs_dir):

  for f in glob.glob(f"{epochs_dir}/sub-*_epo.fif"):
    os.remove(f)

  raws = load_raws(root)
  log = pd.read_csv(log_prep_file, dtype = {"subject": str, "session": str})
  check_log = []

  for key in list(raws):
    sub, ses = key
    if sub in excluded_subjects or (sub, ses) in excluded_sessions:
      continue

    print(f"[{check}] epoching subject {sub}, session {ses}")
    raw = raws.pop(key)

    raw.load_data()
    raw.set_montage(mne.channels.make_standard_montage("biosemi64"))
    raw.filter(l_freq = 2, h_freq = None, method = 'iir',
               iir_params = dict(order = 6, ftype = 'butter'))
    raw.notch_filter(freqs = 50, picks = "eeg", method = "spectrum_fit")

    row = log[(log["subject"] == sub) & (log["session"] == ses)].iloc[0]
    raw.info["bads"] = row["bads"].split(",") if isinstance(row["bads"], str) else []
    raw.interpolate_bads(reset_bads = True)
    raw.set_eeg_reference("average")

    entry = {"subject": sub, "session": ses}

    if check == "ch2_iclabel05":
      ica = mne.preprocessing.read_ica(f"{drive_preprocessed}/sub-{sub}_ses-{ses}_ica.fif")
      labels = label_components(raw, ica, method = "iclabel")
      exclude_labels = ['eye blink', 'muscle artifact', 'heart beat']
      pairs = list(zip(labels["labels"], labels["y_pred_proba"]))
      entry["n_excluded_08"] = sum(lab in exclude_labels and p > 0.8 for lab, p in pairs)
      ica.exclude = [i for i, (lab, p) in enumerate(pairs) if lab in exclude_labels and p > 0.5]
      entry["n_excluded_05"] = len(ica.exclude)
      ica.apply(raw)

    epochs, n_ties = making_epochs(raw, key, answers)
    del raw
    if epochs is None:
      continue

    n_before = epochs.metadata["condition"].value_counts()
    reject = get_rejection_threshold(epochs, ch_types = "eeg", random_state = 123)
    epochs.drop_bad(reject = reject)
    n_after = epochs.metadata["condition"].value_counts()

    entry.update({"threshold_uV": reject["eeg"] * 1e6,
                  "meditation_before": n_before.get("meditation", 0),
                  "mind_wandering_before": n_before.get("mind_wandering", 0),
                  "meditation_after": n_after.get("meditation", 0),
                  "mind_wandering_after": n_after.get("mind_wandering", 0)})

    if len(epochs) > 0:
      if check == "ch3_paper_ica":
        epochs, ica_info = ica_on_epochs(epochs, key, epochs_dir)
        entry.update(ica_info)
      epochs.save(f"{epochs_dir}/sub-{sub}_ses-{ses}_epo.fif", overwrite = True)

    check_log.append(entry)
    del epochs

  pd.DataFrame(check_log).to_csv(f"{epochs_dir}/rejection_log.csv", index = False)

In [ ]:
# Check 1: Standard (Delaunay) neighbors instead of 4.5

info = mne.create_info(ch_names, sfreq = 256, ch_types = "eeg")
info.set_montage("biosemi64")

adjacency_std, adj_names = mne.channels.find_ch_adjacency(info, ch_type = "eeg")
print("Same electrode order:", adj_names == ch_names)

results["ch1_adjacency"] = run_tests(band_power_all, adjacency_std)

In [ ]:
# Checks 2-4:

for check, folder in robust_folders.items():

  epochs_dir = f"{folder}/epochs"
  tf_dir = f"{folder}/tf"

  # 1. epochs
  robustness_epochs(check, epochs_dir)

  # 2. band power
  tf_log_check = []
  for f in sorted(glob.glob(f"{epochs_dir}/sub-*_ses-*_epo.fif")):
    parts = os.path.basename(f).split('_')
    sub = parts[0].replace('sub-', '')
    ses = parts[1].replace('ses-', '')
    if sub in excluded_subjects or (sub, ses) in excluded_sessions:
      continue

    tf_log_check.append(compute_band_power((sub, ses), epochs_dir))

  bp = pd.concat(tf_log_check, ignore_index = True)
  bp.to_csv(f"{tf_dir}/band_power_all.csv", index = False)

  # 3. statistics
  results[check] = run_tests(bp, adjacency)

# Check 2: did the 0.5 threshold remove more components than 0.8?
ch2_log = pd.read_csv(f"{robust_folders['ch2_iclabel05']}/epochs/rejection_log.csv")
print("Sessions with more components removed at 0.5:",
      (ch2_log["n_excluded_05"] > ch2_log["n_excluded_08"]).sum(), "of", len(ch2_log))


In [ ]:
print(participants[participants['subject'].isin(['013', '014', '017', '018'])][['subject', 'group']])

In [ ]:
prep = pd.read_csv(log_prep_file, dtype = {'subject': str, 'session': str})
prep = prep[~prep['subject'].isin(excluded_subjects)]
print(prep[['n_bads', 'n_excluded']].describe().round(1))

In [ ]:
rej = pd.read_csv(rejection_file, dtype = {"subject": str, "session": str})
for c in ["meditation", "mind_wandering"]:
  rej[f"{c}_kept"] = rej[f"{c}_after"] / rej[f"{c}_before"]
print(rej[['meditation_kept', 'mind_wandering_kept']].describe().round(2))

In [ ]:
ch2_log = pd.read_csv(f"{robust_folders['ch2_iclabel05']}/epochs/rejection_log.csv",
                      dtype = {"subject": str, "session": str})
print((ch2_log['n_excluded_05'] > ch2_log['n_excluded_08']).sum(), "of", len(ch2_log))

In [ ]:
# Results without epochs < 3

n_trials = band_power_all.groupby(['subject', 'condition']).size().unstack(fill_value = 0) / 64
low = n_trials[(n_trials < 3).any(axis = 1) & (n_trials > 0).all(axis = 1)].index.tolist()
print("Removed by the minimum-epochs rule:", low)

results["ch5_min_epochs"] = run_tests(band_power_all[~band_power_all['subject'].isin(low)], adjacency)

In [ ]:
for (group, band), r in results["ch5_min_epochs"].items():
  T = r["T_obs"]
  print(group, band, '| cluster p:', np.round(r["cluster_pv"], 3), '| FDR:', r["fdr_channels"],
        '| t_max:', round(T.max(), 2), '| t_FCz:', round(dict(zip(ch_names, T))["FCz"], 2))


In [ ]:
# %% Summary table
rows = []
for analysis, tests in results.items():
  for (group, band), r in tests.items():
    T = r["T_obs"]
    t = dict(zip(ch_names, T))
    rows.append({"analysis": analysis, "group": group, "band": band,
                 "n_clusters": len(r["cluster_pv"]),
                 "min_cluster_p": round(min(r["cluster_pv"]), 3) if len(r["cluster_pv"]) else np.nan,
                 "sig_cluster_channels": r["cluster_channels"],
                 "fdr_channels": r["fdr_channels"],
                 "t_min": round(T.min(), 2), "t_max": round(T.max(), 2),
                 "t_Fz": round(t["Fz"], 2), "t_FCz": round(t["FCz"], 2),
                 "r_with_main": round(np.corrcoef(T, results["main"][(group, band)]["T_obs"])[0, 1], 2)})

summary = pd.DataFrame(rows)
summary.to_csv('/content/drive/MyDrive/Meditation/results_summary.csv', index = False)
print(summary.drop(columns = ["sig_cluster_channels"]).to_string())